# 05 — Q-Interestingness
Composite scores, robustness, permutation control, scalability.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
SEED = 42
np.random.seed(SEED)
for d in ['../outputs/figures','../outputs/tables','../outputs/kernels',
          '../outputs/logs','../data/processed','../data/raw']:
    os.makedirs(d, exist_ok=True)
print("Setup done.")

## Load Kernels & Compute All Scores

In [ ]:
import time
from src.data import load_wine_quality, scale_features_to_pi, get_deterministic_sample, WINE_QUANTUM_FEATURES, WINE_TARGET_COL
from sklearn.preprocessing import StandardScaler
from src.classical import run_isolation_forest
from src.interestingness import quantum_anomaly_score, quantum_novelty, quantum_boundary, q_interestingness, weight_sensitivity_analysis, WEIGHT_CONFIGS
from src.metrics import anomaly_category_labels, top_k_overlap

df = load_wine_quality('../data/raw/winequality-red.csv')
N = 150
idx, df_sample = get_deterministic_sample(df, N, seed=42)
K_q = np.load('../outputs/kernels/K_quantum.npy')

X_q4_std = StandardScaler().fit_transform(df_sample[WINE_QUANTUM_FEATURES].values)
QAS = quantum_anomaly_score(K_q)
CAS = run_isolation_forest(X_q4_std, random_state=42)
N_score = quantum_novelty(K_q, k=5)
B_score = quantum_boundary(K_q)
QI = q_interestingness(QAS, CAS, N_score, B_score, weights=(0.40,0.20,0.20,0.20))
labels = anomaly_category_labels(CAS, QAS)
quality_sample = df_sample[WINE_TARGET_COL].values
print(f"QI: mean={QI.mean():.4f}, max={QI.max():.4f}")

## Weight Sensitivity

In [ ]:
sens = weight_sensitivity_analysis(QAS, CAS, N_score, B_score, WEIGHT_CONFIGS)
corr_mat = sens["spearman_matrix"]
config_names = sens["config_names"]
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(corr_mat, vmin=0, vmax=1, cmap='RdYlGn')
ax.set_xticks(range(len(config_names))); ax.set_yticks(range(len(config_names)))
ax.set_xticklabels(config_names, rotation=15, ha='right')
ax.set_yticklabels(config_names)
for i in range(len(config_names)):
    for j in range(len(config_names)):
        ax.text(j, i, f"{corr_mat[i,j]:.3f}", ha='center', va='center', fontsize=10)
plt.colorbar(im, ax=ax); ax.set_title("Spearman Rank Correlation — Weight Configs")
plt.tight_layout()
plt.savefig('../outputs/figures/09a_weight_sensitivity.png', dpi=120, bbox_inches='tight')
plt.close()
rank_stability = float(np.mean(corr_mat[np.triu_indices(len(config_names),k=1)]))
print(f"Mean Spearman rank correlation (stability): {rank_stability:.4f}")

## Top-20 Q-Interesting Observations

In [ ]:
top20_idx = np.argsort(QI)[::-1][:20]
top20_df = df_sample.iloc[top20_idx][WINE_QUANTUM_FEATURES + [WINE_TARGET_COL]].copy()
top20_df['original_index'] = idx[top20_idx]
top20_df['QAS'] = QAS[top20_idx].round(4)
top20_df['CAS'] = CAS[top20_idx].round(4)
top20_df['quantum_novelty'] = N_score[top20_idx].round(4)
top20_df['quantum_boundary'] = B_score[top20_idx].round(4)
top20_df['QI_balanced'] = QI[top20_idx].round(4)
top20_df['anomaly_category'] = labels[top20_idx]
top20_df.to_csv('../outputs/tables/top_q_interesting_observations.csv', index=False)
print(top20_df.to_string())
print("Saved top_q_interesting_observations.csv")

## Q-Interestingness Map (KPCA space)

In [ ]:
kpca_emb = np.load('../outputs/tables/kpca_emb.npy')
fig, ax = plt.subplots(figsize=(10,8))
sc = ax.scatter(kpca_emb[:,0], kpca_emb[:,1], c=QI, cmap='plasma', s=QI*200+10, alpha=0.8)
plt.colorbar(sc, ax=ax, label='Q-Interestingness')
top10_idx = np.argsort(QI)[::-1][:10]
for i in top10_idx:
    ax.annotate(f"#{i}", (kpca_emb[i,0], kpca_emb[i,1]), fontsize=7, ha='left',
                xytext=(4,4), textcoords='offset points')
ax.set_xlabel('KPCA-1'); ax.set_ylabel('KPCA-2')
ax.set_title('Q-Interestingness Map (Quantum KPCA, N=150)')
plt.tight_layout()
plt.savefig('../outputs/figures/09_q_interestingness_map.png', dpi=120, bbox_inches='tight')
plt.close()
print("Saved 09_q_interestingness_map.png")

## Top-10 Bar Chart

In [ ]:
top10_qi = QI[top10_idx]
top10_labels_idx = [f"obs_{idx[i]}" for i in top10_idx]
fig, ax = plt.subplots(figsize=(10,5))
bars = ax.barh(top10_labels_idx[::-1], top10_qi[::-1],
               color=plt.cm.plasma(top10_qi[::-1]))
ax.set_xlabel('Q-Interestingness Score')
ax.set_title('Top-10 Q-Interesting Observations (balanced weights)')
for bar, val in zip(bars, top10_qi[::-1]):
    ax.text(val+0.005, bar.get_y()+bar.get_height()/2, f"{val:.4f}", va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/figures/10_top_q_interesting.png', dpi=120, bbox_inches='tight')
plt.close()
print("Saved 10_top_q_interesting.png")

## Feature Map Robustness

In [ ]:
from src.quantum import build_zz_feature_map, build_z_feature_map, build_quantum_kernel, compute_kernel_matrix
from src.metrics import centered_kernel_alignment, top_k_overlap
from src.interestingness import quantum_anomaly_score

fmap_rows = []
for fname, fm_fn in [('ZZFeatureMap', lambda: build_zz_feature_map(4, reps=2)),
                      ('ZFeatureMap',  lambda: build_z_feature_map(4, reps=2))]:
    fm = fm_fn()
    qk = build_quantum_kernel(fm)
    K_fm, t = compute_kernel_matrix(qk, scale_features_to_pi(df_sample[WINE_QUANTUM_FEATURES].values))
    cka_fm = centered_kernel_alignment(K_fm, np.load('../outputs/kernels/K_classical.npy'))
    qas_fm = quantum_anomaly_score(K_fm)
    ov10 = top_k_overlap(CAS, qas_fm, 10)
    fmap_rows.append({'feature_map': fname, 'reps': 2, 'runtime_s': round(t,2),
                      'kernel_mean': round(float(K_fm.mean()),4), 'kernel_std': round(float(K_fm.std()),4),
                      'cka': round(cka_fm,4), 'top10_anomaly_overlap': round(ov10,4)})
    print(f"  {fname}: CKA={cka_fm:.4f}, top10_overlap={ov10:.4f}, t={t:.1f}s")
pd.DataFrame(fmap_rows).to_csv('../outputs/tables/feature_map_comparison.csv', index=False)
print("Saved feature_map_comparison.csv")

## Circuit Depth Robustness

In [ ]:
depth_rows = []
for reps in [1, 2, 3]:
    fm = build_zz_feature_map(4, reps=reps)
    qk = build_quantum_kernel(fm)
    K_r, t = compute_kernel_matrix(qk, scale_features_to_pi(df_sample[WINE_QUANTUM_FEATURES].values))
    cka_r = centered_kernel_alignment(K_r, np.load('../outputs/kernels/K_classical.npy'))
    qas_r = quantum_anomaly_score(K_r)
    ov10 = top_k_overlap(CAS, qas_r, 10)
    depth_rows.append({'reps': reps, 'circuit_depth': fm.decompose().depth(),
                       'runtime_s': round(t,2), 'kernel_mean': round(float(K_r.mean()),4),
                       'kernel_std': round(float(K_r.std()),4), 'cka': round(cka_r,4),
                       'top10_anomaly_overlap': round(ov10,4)})
    print(f"  reps={reps}: CKA={cka_r:.4f}, top10={ov10:.4f}, t={t:.1f}s")
pd.DataFrame(depth_rows).to_csv('../outputs/tables/depth_robustness.csv', index=False)
print("Saved depth_robustness.csv")

## Permutation Control

In [ ]:
from src.metrics import centered_kernel_alignment
perm_rows = []
for seed in range(10):
    rng = np.random.RandomState(seed)
    X_perm = df_sample[WINE_QUANTUM_FEATURES].values.copy()
    for col_i in range(X_perm.shape[1]):
        X_perm[:, col_i] = rng.permutation(X_perm[:, col_i])
    X_perm_scaled = scale_features_to_pi(X_perm)
    fm_p = build_zz_feature_map(4, reps=2)
    qk_p = build_quantum_kernel(fm_p)
    K_p, _ = compute_kernel_matrix(qk_p, X_perm_scaled)
    cka_p = centered_kernel_alignment(K_p, np.load('../outputs/kernels/K_classical.npy'))
    perm_rows.append({'seed': seed, 'permutation_cka': round(cka_p,4)})
    print(f"  seed={seed}: perm_CKA={cka_p:.4f}")
perm_df = pd.DataFrame(perm_rows)
observed_cka = centered_kernel_alignment(K_q, np.load('../outputs/kernels/K_classical.npy'))
perm_mean = perm_df['permutation_cka'].mean()
perm_std  = perm_df['permutation_cka'].std()
z_score   = (observed_cka - perm_mean) / perm_std if perm_std > 0 else float('nan')
perm_df.loc[len(perm_df)] = {'seed': 'observed', 'permutation_cka': round(observed_cka,4)}
perm_df.loc[len(perm_df)] = {'seed': 'perm_mean', 'permutation_cka': round(perm_mean,4)}
perm_df.loc[len(perm_df)] = {'seed': 'perm_std',  'permutation_cka': round(perm_std,4)}
perm_df.loc[len(perm_df)] = {'seed': 'z_score',   'permutation_cka': round(z_score,4)}
perm_df.to_csv('../outputs/tables/permutation_control.csv', index=False)
print(f"Observed CKA={observed_cka:.4f}, perm mean={perm_mean:.4f}, z={z_score:.2f}")
print("Saved permutation_control.csv")

## Scalability Experiment

In [ ]:
scale_rows = []
for n_obs in [50, 100, 150, 200]:
    rng = np.random.RandomState(42)
    s_idx = np.sort(rng.choice(len(df), n_obs, replace=False))
    X_s = scale_features_to_pi(df.iloc[s_idx][WINE_QUANTUM_FEATURES].values)
    t0 = time.time()
    fm_s = build_zz_feature_map(4, reps=2)
    qk_s = build_quantum_kernel(fm_s)
    K_s, _ = compute_kernel_matrix(qk_s, X_s)
    elapsed_s = time.time() - t0
    scale_rows.append({'n': n_obs, 'runtime_s': round(elapsed_s,2),
                       'kernel_mean': round(float(K_s.mean()),4), 'kernel_std': round(float(K_s.std()),4)})
    print(f"  N={n_obs}: {elapsed_s:.1f}s")
pd.DataFrame(scale_rows).to_csv('../outputs/tables/scalability.csv', index=False)
sr = pd.DataFrame(scale_rows)
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(sr['n'], sr['runtime_s'], 'o-', color='steelblue')
ax.set_xlabel('Sample Size (N)'); ax.set_ylabel('Runtime (s)')
ax.set_title('Scalability: Sample Size vs Runtime (ZZFeatureMap, reps=2)')
plt.tight_layout()
plt.savefig('../outputs/figures/11_scalability.png', dpi=120, bbox_inches='tight')
plt.close()
print("Saved scalability.csv and scalability figure.")
print("Notebook 05 complete.")